<a href="https://colab.research.google.com/github/AhmedHamedAttia/HEAL-IoMT-IDS/blob/main/HEAL_R2_Clean_Reproducible_Experiments_v3_RAM_SAFE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HEAL-R2 — RAM-Safe Reproducible Experimental Pipeline (v3.0)

This notebook supersedes v2. It processes and loads one dataset at a time, streams CICIoMT sampling from Parquet, checkpoints every experiment, and caps classifier-transfer training to prevent Colab RAM exhaustion.


In [ ]:
# CELL 1/9 — Environment, Drive, versioned configuration
!pip -q install kagglehub lightgbm xgboost pyarrow openpyxl psutil scipy joblib

from google.colab import drive
drive.mount('/content/drive')

import os, json, time, math, random, hashlib, warnings, shutil, zipfile, re, gc
from pathlib import Path
import numpy as np, pandas as pd, joblib
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score, matthews_corrcoef,
    balanced_accuracy_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import psutil
warnings.filterwarnings('ignore')

ROOT=Path('/content/drive/MyDrive/HEAL_R2')
RUN=ROOT/'v3_ram_safe'
DIR={k:RUN/k for k in ['data_processed','checkpoints','results','figures','models','logs','final']}
RAW=ROOT/'data_raw'  # reuse already downloaded raw data
for p in DIR.values(): p.mkdir(parents=True,exist_ok=True)

CFG={
 'version':'HEAL_R2_clean_v3.0_ram_safe', 'seeds':[11,22,33,44,55],
 'test_size':0.20, 'val_size_from_train':0.20,
 # Bottleneck fix: wrapper optimization uses a small fixed subset; final evaluation uses all non-test data.
 'fs_train_cap':4000, 'fs_val_cap':2500,
 'transfer_train_cap':50000,
 'population':10, 'iterations':8, 'navigators':3,
 'tiny_hidden':10, 'tiny_fs_max_iter':30, 'tiny_final_max_iter':150,
 'fitness_weights':{'recall':0.55,'f1':0.30,'compactness':0.15},
 'tau_grid':[0.05,0.10,0.15,0.20,0.25], 'drift_window':1000, 'drift_consecutive':2
}
(RUN/'config.json').write_text(json.dumps(CFG,indent=2))

def seed_all(s): random.seed(s); np.random.seed(s); os.environ['PYTHONHASHSEED']=str(s)
def atomic_joblib(obj,p):
 p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); tmp=Path(str(p)+'.tmp'); joblib.dump(obj,tmp,compress=3); os.replace(tmp,p)
def atomic_csv(df,p):
 p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); tmp=Path(str(p)+'.tmp'); df.to_csv(tmp,index=False); os.replace(tmp,p)
def class_counts(y): return {int(k):int(v) for k,v in zip(*np.unique(y,return_counts=True))}
print(json.dumps(CFG,indent=2)); print('CPU',psutil.cpu_count(),'RAM GB',round(psutil.virtual_memory().total/1e9,1))
print('Workspace:',RUN)

DATASETS=['iot_flock','wustl_ehms_2020','ciciomt2024']
def processed_path(ds): return DIR['data_processed']/f'{ds}.joblib'
def load_ds(ds):
 d=joblib.load(processed_path(ds)); return d
def ram(msg='RAM'):
 gc.collect(); print(msg, round(psutil.Process(os.getpid()).memory_info().rss/1e9,2),'GB process | available',round(psutil.virtual_memory().available/1e9,2),'GB')


Mounted at /content/drive
{
  "version": "HEAL_R2_clean_v3.0_ram_safe",
  "seeds": [
    11,
    22,
    33,
    44,
    55
  ],
  "test_size": 0.2,
  "val_size_from_train": 0.2,
  "fs_train_cap": 4000,
  "fs_val_cap": 2500,
  "transfer_train_cap": 50000,
  "population": 10,
  "iterations": 8,
  "navigators": 3,
  "tiny_hidden": 10,
  "tiny_fs_max_iter": 30,
  "tiny_final_max_iter": 150,
  "fitness_weights": {
    "recall": 0.55,
    "f1": 0.3,
    "compactness": 0.15
  },
  "tau_grid": [
    0.05,
    0.1,
    0.15,
    0.2,
    0.25
  ],
  "drift_window": 1000,
  "drift_consecutive": 2
}
CPU 2 RAM GB 13.6
Workspace: /content/drive/MyDrive/HEAL_R2/v3_ram_safe


In [ ]:
# CELL 2/9 — Acquire/verify raw datasets (cached downloads are reused)
import kagglehub, requests

def files_under(p): return [x for x in Path(p).rglob('*') if x.is_file() and x.stat().st_size>0]
def kaggle_fetch(slug,dest):
 dest=Path(dest); dest.mkdir(parents=True,exist_ok=True)
 if files_under(dest): print('[CACHE]',slug); return
 src=Path(kagglehub.dataset_download(slug))
 for f in src.rglob('*'):
  if f.is_file():
   out=dest/f.relative_to(src); out.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(f,out)

kaggle_fetch('faisalmalik/iot-healthcare-security-dataset',RAW/'iot_flock')
try: kaggle_fetch('limamateus/cic-iomt-2024-wifi-mqtt',RAW/'ciciomt2024')
except Exception: kaggle_fetch('amineipad/cic-iomt-dataset-2024',RAW/'ciciomt2024')

wdir=RAW/'wustl_ehms_2020'; wdir.mkdir(parents=True,exist_ok=True)
if not files_under(wdir):
 page='https://www.cs.wustl.edu/~jain/ehms/index.html'; html=requests.get(page,timeout=30).text
 links=re.findall(r'href=["\']([^"\']+)["\']',html,re.I)
 from urllib.parse import urljoin
 for rel in links:
  if any(rel.lower().endswith(x) for x in ['.csv','.zip','.gz']):
   try:
    r=requests.get(urljoin(page,rel),timeout=120)
    if r.ok and len(r.content)>10000: (wdir/Path(rel).name).write_bytes(r.content); break
   except: pass
for base in [RAW/'iot_flock',RAW/'ciciomt2024',wdir]:
 for z in base.rglob('*.zip'):
  out=z.parent/(z.stem+'_extracted')
  if not out.exists(): out.mkdir(); zipfile.ZipFile(z).extractall(out)
for ds in ['iot_flock','wustl_ehms_2020','ciciomt2024']:
 print(ds,[(x.name,round(x.stat().st_size/1e6,1)) for x in files_under(RAW/ds) if x.suffix.lower() in ['.csv','.parquet']][:12])


[CACHE] faisalmalik/iot-healthcare-security-dataset
[CACHE] limamateus/cic-iomt-2024-wifi-mqtt
iot_flock [('environmentMonitoring.csv', 8.9), ('Attack.csv', 77.2), ('patientMonitoring.csv', 21.7)]
wustl_ehms_2020 [('wustl-ehms-2020_with_attacks_categories.csv', 3.9)]
ciciomt2024 [('CIC_IoMT_2024_WiFi_MQTT_test.csv', 377.7), ('CIC_IoMT_2024_WiFi_MQTT_test.parquet', 22.1), ('CIC_IoMT_2024_WiFi_MQTT_train.csv', 1694.6), ('CIC_IoMT_2024_WiFi_MQTT_train.parquet', 123.9)]


In [ ]:
# CELL 3/9 — RAM-SAFE preprocessing + automatic leakage audit
# Only ONE dataset is held in RAM at a time. CICIoMT Parquet is sampled in streaming batches.
import pyarrow.parquet as pq
ID_WORDS=['flow id','src ip','source ip','dst ip','destination ip','timestamp','src mac','dst mac','mac address']
LEAK_WORDS=['label','attack_type','attack type','attack category','attack_category','class']

def readtab(p,nrows=None):
 p=Path(p)
 if p.suffix.lower()=='.parquet': return pd.read_parquet(p) if nrows is None else pd.read_parquet(p).head(nrows)
 return pd.read_csv(p,low_memory=False,nrows=nrows)
def tables(ds): return sorted(list((RAW/ds).rglob('*.parquet'))+list((RAW/ds).rglob('*.csv')),key=lambda p:p.stat().st_size,reverse=True)
def norm_label(v): return str(v).strip().lower().replace('_train','').replace('_test','').replace('-train','').replace('-test','')
def binary_labels(s):
 benign={'0','benign','normal','normaltraffic','normal traffic','non-attack','no attack','false'}
 return np.array([0 if norm_label(v) in benign else 1 for v in s],dtype=np.int8)
def clean_numeric(X):
 X=X.copy(); drop=[]
 for c in X.columns:
  lc=str(c).strip().lower()
  if lc in ID_WORDS or any(w in lc for w in LEAK_WORDS): drop.append(c)
 X=X.drop(columns=drop,errors='ignore')
 for c in list(X.columns):
  if not pd.api.types.is_numeric_dtype(X[c]):
   z=pd.to_numeric(X[c],errors='coerce')
   if z.notna().mean()>=.95: X[c]=z
   else: X=X.drop(columns=[c])
 X=X.replace([np.inf,-np.inf],np.nan); X=X.loc[:,X.notna().mean()>=.20]; X=X.loc[:,X.nunique(dropna=True)>1]
 return X.astype('float32')
def transform_split(Xtr,Xv,Xte):
 imp=SimpleImputer(strategy='median'); sc=StandardScaler()
 A=imp.fit_transform(Xtr); B=imp.transform(Xv); C=imp.transform(Xte)
 return sc.fit_transform(A).astype('float32'),sc.transform(B).astype('float32'),sc.transform(C).astype('float32'),imp,sc

def prep_iot(seed=11):
 parts=[]
 for f in tables('iot_flock'):
  if f.name.lower() in ['attack.csv','patientmonitoring.csv','environmentmonitoring.csv']:
   d=readtab(f); d['__file']=f.name; parts.append(d)
 if not parts: raise RuntimeError('IoT-Flock expected files not found')
 df=pd.concat(parts,ignore_index=True); del parts; gc.collect(); label='label' if 'label' in df else 'Label'
 y=binary_labels(df[label]); X=clean_numeric(df.drop(columns=[label,'__file'],errors='ignore')); del df; gc.collect()
 idx=np.arange(len(y)); tr,te=train_test_split(idx,test_size=CFG['test_size'],stratify=y,random_state=seed); tr,va=train_test_split(tr,test_size=CFG['val_size_from_train'],stratify=y[tr],random_state=seed)
 A,B,C,imp,sc=transform_split(X.iloc[tr],X.iloc[va],X.iloc[te]); features=list(X.columns); del X; gc.collect()
 return dict(X_train=A,y_train=y[tr],X_val=B,y_val=y[va],X_test=C,y_test=y[te],features=features,label_col=label,official_test_split=False,imputer=imp,scaler=sc)

def prep_wustl(seed=11):
 fs=tables('wustl_ehms_2020'); f=next((x for x in fs if 'attack' in x.name.lower()),fs[0]); df=readtab(f)
 label='Label' if 'Label' in df else 'label'; y=np.asarray(pd.to_numeric(df[label],errors='coerce').fillna(0).astype(int),dtype=np.int8); X=clean_numeric(df.drop(columns=[label],errors='ignore')); del df; gc.collect()
 idx=np.arange(len(y)); tr,te=train_test_split(idx,test_size=CFG['test_size'],stratify=y,random_state=seed); tr,va=train_test_split(tr,test_size=CFG['val_size_from_train'],stratify=y[tr],random_state=seed)
 A,B,C,imp,sc=transform_split(X.iloc[tr],X.iloc[va],X.iloc[te]); features=list(X.columns); del X; gc.collect()
 return dict(X_train=A,y_train=y[tr],X_val=B,y_val=y[va],X_test=C,y_test=y[te],features=features,label_col=label,official_test_split=False,imputer=imp,scaler=sc)

def stream_stratified_parquet(path,total_cap,seed,batch_size=100000):
 pf=pq.ParquetFile(path); names=pf.schema_arrow.names; label='label' if 'label' in names else 'Label'
 # pass 1: class counts using label only
 counts=np.zeros(2,dtype=np.int64)
 for batch in pf.iter_batches(batch_size=batch_size,columns=[label]):
  y=binary_labels(batch.column(0).to_pylist()); counts += np.bincount(y,minlength=2)
 target=np.floor(total_cap*counts/counts.sum()).astype(int); target[1]=total_cap-target[0]
 print(' streaming',Path(path).name,'raw counts',counts.tolist(),'target',target.tolist())
 # pass 2: priority reservoir by class; keeps smallest random keys => uniform fixed-size sample
 rng=np.random.default_rng(seed); res={0:None,1:None}; keys={0:np.empty(0),1:np.empty(0)}
 for batch in pf.iter_batches(batch_size=batch_size):
  df=batch.to_pandas(); y=binary_labels(df[label]);
  for cls in (0,1):
   ids=np.flatnonzero(y==cls)
   if not len(ids): continue
   part=df.iloc[ids].copy(); kval=rng.random(len(part))
   if res[cls] is None: comb=part; kk=kval
   else: comb=pd.concat([res[cls],part],ignore_index=True); kk=np.r_[keys[cls],kval]
   if len(comb)>target[cls]:
    take=np.argpartition(kk,target[cls]-1)[:target[cls]]; comb=comb.iloc[take].reset_index(drop=True); kk=kk[take]
   res[cls],keys[cls]=comb,kk
  del df; gc.collect()
 out=pd.concat([res[0],res[1]],ignore_index=True).sample(frac=1,random_state=seed).reset_index(drop=True)
 return out,label

def prep_cic(seed=11):
 fs=tables('ciciomt2024'); trainf=next((f for f in fs if 'train' in f.name.lower()),None); testf=next((f for f in fs if 'test' in f.name.lower()),None)
 if trainf is None or testf is None: raise RuntimeError('CICIoMT official train/test files not found')
 if trainf.suffix.lower()=='.parquet' and testf.suffix.lower()=='.parquet':
  trdf,label=stream_stratified_parquet(trainf,200000,seed); tedf,_=stream_stratified_parquet(testf,50000,seed+1)
 else:
  # CSV fallback: chunk-safe reservoir is not needed for the current cached source; fail clearly rather than OOM silently.
  raise RuntimeError('CICIoMT RAM-safe pipeline expects the downloaded Parquet train/test files.')
 ytr0=binary_labels(trdf[label]); yte=binary_labels(tedf[label]); Xtr0=clean_numeric(trdf.drop(columns=[label],errors='ignore')); Xte0=clean_numeric(tedf.drop(columns=[label],errors='ignore')); del trdf,tedf; gc.collect()
 common=[c for c in Xtr0.columns if c in Xte0.columns]; Xtr0,Xte0=Xtr0[common],Xte0[common]
 ids=np.arange(len(ytr0)); tr,va=train_test_split(ids,test_size=CFG['val_size_from_train'],stratify=ytr0,random_state=seed)
 A,B,C,imp,sc=transform_split(Xtr0.iloc[tr],Xtr0.iloc[va],Xte0); del Xtr0,Xte0; gc.collect()
 return dict(X_train=A,y_train=ytr0[tr],X_val=B,y_val=ytr0[va],X_test=C,y_test=yte,features=common,label_col=label,official_test_split=True,imputer=imp,scaler=sc)

PREP={'iot_flock':prep_iot,'wustl_ehms_2020':prep_wustl,'ciciomt2024':prep_cic}
legacy=ROOT/'v2_clean'/'data_processed'
for ds,fn in PREP.items():
 p=processed_path(ds); old=legacy/f'{ds}.joblib'
 if p.exists(): print('[CACHE v3]',ds)
 elif old.exists(): shutil.copy2(old,p); print('[REUSED validated v2 cache]',ds)
 else:
  print('[PREPROCESS]',ds); d=fn(); atomic_joblib(d,p); del d; gc.collect(); print('[SAVED]',ds)
 d=load_ds(ds); assert set(np.unique(d['y_train']))=={0,1} and not np.isnan(d['X_train']).any()
 print(ds,len(d['features']),d['X_train'].shape,d['X_val'].shape,d['X_test'].shape,class_counts(d['y_train']),'official_test=',d['official_test_split']); del d; ram(' after '+ds)

# Audit one dataset at a time; never keep all datasets resident.
audit=[]
for ds in DATASETS:
 d=load_ds(ds)
 for j,f in enumerate(d['features']):
  try: a=roc_auc_score(d['y_train'],d['X_train'][:,j]); a=max(a,1-a)
  except: a=np.nan
  audit.append({'dataset':ds,'index':j,'feature':f,'univariate_auc':a,'name_suspicious':any(w in str(f).lower() for w in LEAK_WORDS)})
 del d; gc.collect()
A=pd.DataFrame(audit); atomic_csv(A,DIR['results']/'pre_experiment_feature_audit.csv')
display(A.sort_values(['dataset','univariate_auc'],ascending=[True,False]).groupby('dataset').head(12))
if A.name_suspicious.any(): raise RuntimeError('Potential target-derived feature survived preprocessing; inspect audit before continuing.')
print('[OK] RAM-safe preprocessing/audit complete.'); ram()


[REUSED validated v2 cache] iot_flock
iot_flock 30 (120764, 30) (30191, 30) (37739, 30) {0: 69483, 1: 51281} official_test= False
 after iot_flock 0.33 GB process | available 12.36 GB
[REUSED validated v2 cache] wustl_ehms_2020
wustl_ehms_2020 31 (10443, 31) (2611, 31) (3264, 31) {0: 9133, 1: 1310} official_test= False
 after wustl_ehms_2020 0.33 GB process | available 12.34 GB
[PREPROCESS] ciciomt2024


RuntimeError: CICIoMT RAM-safe pipeline expects the downloaded Parquet train/test files.

In [ ]:
# CELL 4/9 — Faithful baseline definitions + proposed TW²O
# Controlled set: MI, L1, GA, PSO, DE, GWO, BWOA, TW2O. No invented WWO/BDA/ACO approximations.
def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-20,20)))
def nonempty(mask,rng):
 mask=np.asarray(mask,dtype=np.int8).copy()
 if mask.sum()==0: mask[rng.integers(len(mask))]=1
 return mask

def balanced_cap(X,y,n,seed):
 rng=np.random.default_rng(seed); ids0=np.where(y==0)[0]; ids1=np.where(y==1)[0]; k=min(len(ids0),len(ids1),n//2)
 ids=np.r_[rng.choice(ids0,k,False),rng.choice(ids1,k,False)]; rng.shuffle(ids); return X[ids],y[ids]
def natural_cap(X,y,n,seed):
 if len(y)<=n:return X,y
 ids,_=train_test_split(np.arange(len(y)),train_size=n,stratify=y,random_state=seed); return X[ids],y[ids]

class Evaluator:
 def __init__(self,d,seed):
  self.Xt,self.yt=balanced_cap(d['X_train'],d['y_train'],CFG['fs_train_cap'],seed)
  self.Xv,self.yv=natural_cap(d['X_val'],d['y_val'],CFG['fs_val_cap'],seed+1000)
  self.m=self.Xt.shape[1]; self.seed=seed; self.cache={}; self.evals=0
 def eval(self,mask):
  mask=nonempty(mask,np.random.default_rng(self.seed)); key=mask.tobytes()
  if key in self.cache:return self.cache[key]
  ix=np.flatnonzero(mask); clf=MLPClassifier(hidden_layer_sizes=(CFG['tiny_hidden'],),activation='relu',max_iter=CFG['tiny_fs_max_iter'],early_stopping=True,n_iter_no_change=4,batch_size=128,random_state=self.seed)
  t=time.perf_counter()
  try:
   clf.fit(self.Xt[:,ix],self.yt); p=clf.predict(self.Xv[:,ix]); rec=recall_score(self.yv,p,zero_division=0); f1=f1_score(self.yv,p,zero_division=0); comp=1-len(ix)/self.m
   w=CFG['fitness_weights']; fit=w['recall']*rec+w['f1']*f1+w['compactness']*comp
   z={'fitness':float(fit),'recall':float(rec),'f1':float(f1),'compactness':float(comp),'selected':len(ix),'seconds':time.perf_counter()-t}
  except Exception as e:z={'fitness':-1e9,'recall':0.,'f1':0.,'compactness':0.,'selected':len(ix),'seconds':np.nan,'error':repr(e)}
  self.cache[key]=z; self.evals+=1; return z

def best_of(pop,E):
 infos=[E.eval(x) for x in pop]; scores=np.array([z['fitness'] for z in infos]); i=int(scores.argmax()); return scores,infos,i

def run_selector(method,d,seed):
 rng=np.random.default_rng(seed); E=Evaluator(d,seed); m=E.m; N=CFG['population']; T=CFG['iterations']; start=time.perf_counter()
 if method=='MI':
  s=mutual_info_classif(E.Xt,E.yt,random_state=seed); ranking=np.argsort(s)[::-1]; cand=[]
  for q in [.10,.20,.30,.40,.50,.70,1.0]:
   k=max(1,round(m*q)); z=np.zeros(m,dtype=np.int8); z[ranking[:k]]=1; cand.append((E.eval(z)['fitness'],z))
  fit,b=max(cand,key=lambda x:x[0]); return b,E.eval(b),[fit],E.evals,time.perf_counter()-start
 if method=='L1':
  cand=[]
  for C in [.001,.01,.05,.1,.5,1.0]:
   q=LogisticRegression(penalty='l1',solver='liblinear',C=C,class_weight='balanced',max_iter=1000,random_state=seed).fit(E.Xt,E.yt)
   z=nonempty((np.abs(q.coef_).ravel()>1e-8).astype(np.int8),rng); cand.append((E.eval(z)['fitness'],z))
  fit,b=max(cand,key=lambda x:x[0]); return b,E.eval(b),[fit],E.evals,time.perf_counter()-start
 P=rng.integers(0,2,(N,m),dtype=np.int8); P=np.array([nonempty(x,rng) for x in P]); V=np.zeros((N,m)); pbest=P.copy(); pbestf=np.full(N,-np.inf); hist=[]; best=None; bestinfo=None
 # continuous states for canonical continuous-update families
 pos=rng.uniform(-2,2,(N,m)) if method in ['GWO','BWOA'] else None
 if pos is not None:P=np.array([nonempty((rng.random(m)<sigmoid(x)).astype(np.int8),rng) for x in pos])
 for t in range(T):
  scores,infos,bi=best_of(P,E)
  if bestinfo is None or scores[bi]>bestinfo['fitness']: best=P[bi].copy(); bestinfo=infos[bi].copy(); bestpos=pos[bi].copy() if pos is not None else None
  hist.append(bestinfo['fitness']); order=np.argsort(scores)[::-1]; a=2*(1-t/max(1,T-1))
  if method=='GA':
   elite=P[order[:max(2,N//4)]]; new=[best.copy()]
   while len(new)<N:
    p1,p2=elite[rng.integers(len(elite))],elite[rng.integers(len(elite))]; z=np.where(rng.random(m)<.5,p1,p2).astype(np.int8); mut=rng.random(m)<1/m; z[mut]=1-z[mut]; new.append(nonempty(z,rng))
   P=np.array(new)
  elif method=='PSO':
   for i in range(N):
    if scores[i]>pbestf[i]:pbestf[i]=scores[i];pbest[i]=P[i].copy()
   w=.9-.5*t/max(1,T-1); V=w*V+2*rng.random((N,m))*(pbest-P)+2*rng.random((N,m))*(best-P); P=np.array([nonempty((rng.random(m)<sigmoid(v)).astype(np.int8),rng) for v in V])
  elif method=='DE':
   new=P.copy()
   for i in range(N):
    ids=rng.choice(np.delete(np.arange(N),i),3,False); mutant=P[ids[0]].astype(float)+.5*(P[ids[1]]-P[ids[2]]); mb=nonempty((rng.random(m)<sigmoid(mutant)).astype(np.int8),rng); cross=rng.random(m)<.7; cross[rng.integers(m)]=1; trial=nonempty(np.where(cross,mb,P[i]).astype(np.int8),rng)
    if E.eval(trial)['fitness']>scores[i]:new[i]=trial
   P=new
  elif method=='GWO':
   leaders=pos[order[:3]]; new=np.zeros_like(pos)
   for i in range(N):
    xs=[]
    for L in leaders:
     A=2*a*rng.random(m)-a; C=2*rng.random(m); xs.append(L-A*np.abs(C*L-pos[i]))
    new[i]=np.mean(xs,axis=0)
   pos=np.clip(new,-8,8); P=np.array([nonempty((rng.random(m)<sigmoid(x)).astype(np.int8),rng) for x in pos])
  elif method=='BWOA':
   new=pos.copy()
   for i in range(N):
    A=2*a*rng.random()-a; C=2*rng.random(); p=rng.random()
    if p<.5:
     ref=bestpos if abs(A)<1 else pos[rng.integers(N)]; z=ref-A*np.abs(C*ref-pos[i])
    else:
     l=rng.uniform(-1,1); z=np.abs(bestpos-pos[i])*np.exp(l)*np.cos(2*np.pi*l)+bestpos
    new[i]=np.clip(z,-8,8)
   pos=new; P=np.array([nonempty((rng.random(m)<sigmoid(x)).astype(np.int8),rng) for x in pos])
  elif method=='TW2O':
   nav=P[order[:min(CFG['navigators'],N)]]; progress=t/max(1,T-1); mutation=.02+.18*(1-progress); new=[best.copy()]
   while len(new)<N:
    follower=P[rng.integers(N)].copy(); leader=nav[rng.integers(len(nav))]; z=follower.copy(); q=rng.random(m)<(.30+.25*progress); z[q]=leader[q]; q=rng.random(m)<(.15+.35*progress); z[q]=best[q]; q=rng.random(m)<mutation; z[q]=1-z[q]; new.append(nonempty(z,rng))
   P=np.array(new)
  else: raise ValueError(method)
 return best,bestinfo,hist,E.evals,time.perf_counter()-start

METHODS=['MI','L1','GA','PSO','DE','GWO','BWOA','TW2O']
# Tiny smoke test only
d_smoke=load_ds('iot_flock')
for meth in ['MI','BWOA','TW2O']:
 b,z,h,n,sec=run_selector(meth,d_smoke,11); print(meth,'selected',int(b.sum()),'fitness',round(z['fitness'],4),'evals',n,'sec',round(sec,1))
del d_smoke; gc.collect(); ram('after smoke test')
print('[OK] selector engine ready:',METHODS)


In [ ]:
# CELL 5/9 — Checkpointed multi-seed feature-selection benchmark + final TinyML test metrics
FS=DIR['checkpoints']/'feature_selection'; FS.mkdir(exist_ok=True)
def metrics(y,p,s=None):
 tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel(); out={'accuracy':accuracy_score(y,p),'precision':precision_score(y,p,zero_division=0),'recall':recall_score(y,p,zero_division=0),'f1':f1_score(y,p,zero_division=0),'specificity':tn/(tn+fp) if tn+fp else np.nan,'fpr':fp/(fp+tn) if fp+tn else np.nan,'fnr':fn/(fn+tp) if fn+tp else np.nan,'mcc':matthews_corrcoef(y,p),'balanced_accuracy':balanced_accuracy_score(y,p),'tn':tn,'fp':fp,'fn':fn,'tp':tp}
 if s is not None:
  out['roc_auc']=roc_auc_score(y,s); out['pr_auc']=average_precision_score(y,s)
 return {k:(float(v) if isinstance(v,(float,np.floating)) else int(v)) for k,v in out.items()}
def final_tiny(d,mask,seed):
 ix=np.flatnonzero(mask); X=np.r_[d['X_train'][:,ix],d['X_val'][:,ix]]; y=np.r_[d['y_train'],d['y_val']]
 clf=MLPClassifier(hidden_layer_sizes=(CFG['tiny_hidden'],),max_iter=CFG['tiny_final_max_iter'],early_stopping=True,n_iter_no_change=10,batch_size=128,random_state=seed)
 t=time.perf_counter(); clf.fit(X,y); train=time.perf_counter()-t; t=time.perf_counter(); p=clf.predict(d['X_test'][:,ix]); infer=time.perf_counter()-t; s=clf.predict_proba(d['X_test'][:,ix])[:,1]
 z=metrics(d['y_test'],p,s); z.update({'train_s':train,'inference_ms_per_sample':1000*infer/len(p),'approx_parameters':len(ix)*CFG['tiny_hidden']+2*CFG['tiny_hidden']+1}); return z

rows=[]; total=len(DATASETS)*len(METHODS)*len(CFG['seeds']); done=0; t0=time.perf_counter()
for ds in DATASETS:
 d=load_ds(ds); ram('loaded '+ds)
 for method in METHODS:
  for seed in CFG['seeds']:
   done+=1; fp=FS/f'{ds}__{method}__seed{seed}.joblib'
   if fp.exists(): res=joblib.load(fp); print(f'[{done}/{total}] CACHE',fp.name)
   else:
    print(f'[{done}/{total}] RUN',ds,method,seed); seed_all(seed); mask,info,hist,evals,secs=run_selector(method,d,seed); test=final_tiny(d,mask,seed)
    res={'dataset':ds,'method':method,'seed':seed,'mask':mask,'selected':int(mask.sum()),'reduction':float(1-mask.sum()/len(mask)),'selector':info,'history':hist,'fitness_evals':evals,'selector_s':secs,'test':test,'features':[d['features'][i] for i in np.flatnonzero(mask)],'protocol_version':CFG['version']}; atomic_joblib(res,fp)
    print(' selected',res['selected'],'fitness',round(info['fitness'],4),'test F1',round(test['f1'],4),'MCC',round(test['mcc'],4),'sec',round(secs,1))
   rows.append({'dataset':ds,'method':method,'seed':seed,'selected':res['selected'],'reduction':res['reduction'],'selector_fitness':res['selector']['fitness'],'selector_s':res['selector_s'],**res['test']})
   gc.collect()
 del d; gc.collect(); ram('released '+ds)
R=pd.DataFrame(rows); atomic_csv(R,DIR['results']/'feature_selection_all_runs.csv')
S=R.groupby(['dataset','method']).agg(['mean','std']); S.to_csv(DIR['results']/'feature_selection_summary_mean_std.csv')
display(R.groupby(['dataset','method'])[['selected','reduction','accuracy','recall','f1','specificity','mcc','pr_auc','selector_s']].agg(['mean','std']).round(4))
print('Elapsed min',round((time.perf_counter()-t0)/60,1),'— rerunning this cell safely resumes from checkpoints.')


In [ ]:
# CELL 6/9 — Classifier-transfer validation + core detection ablation
# Reviewer concern: selected features must not be validated only with the TinyML evaluator.
CLFS={
 'TinyML-MLP':lambda s:MLPClassifier(hidden_layer_sizes=(10,),max_iter=150,early_stopping=True,random_state=s),
 'LogReg':lambda s:LogisticRegression(max_iter=1000,class_weight='balanced',random_state=s),
 'RF':lambda s:RandomForestClassifier(n_estimators=200,n_jobs=-1,class_weight='balanced',random_state=s),
 'ExtraTrees':lambda s:ExtraTreesClassifier(n_estimators=200,n_jobs=-1,class_weight='balanced',random_state=s),
 'SVM':lambda s:SVC(C=2,kernel='rbf',probability=True,class_weight='balanced',random_state=s),
 'XGBoost':lambda s:XGBClassifier(n_estimators=200,max_depth=5,learning_rate=.08,subsample=.8,colsample_bytree=.8,n_jobs=-1,random_state=s,eval_metric='logloss'),
 'LightGBM':lambda s:LGBMClassifier(n_estimators=200,learning_rate=.08,n_jobs=-1,random_state=s,verbosity=-1)
}
TR=DIR['checkpoints']/'classifier_transfer'; TR.mkdir(exist_ok=True); rows=[]
# Use each seed's TW2O subset; compare full features vs TW2O subset for every classifier.
for ds in DATASETS:
 d=load_ds(ds); ram('transfer loaded '+ds)
 for seed in CFG['seeds']:
  tw=joblib.load(FS/f'{ds}__TW2O__seed{seed}.joblib')['mask'].astype(bool)
  for subset,mask in [('All',np.ones(len(d['features']),bool)),('TW2O',tw)]:
   ix=np.flatnonzero(mask); Xall=np.concatenate((d['X_train'][:,ix],d['X_val'][:,ix]),axis=0); yall=np.concatenate((d['y_train'],d['y_val'])); X,y=natural_cap(Xall,yall,CFG['transfer_train_cap'],seed+7000); del Xall,yall; gc.collect()
   for name,maker in CLFS.items():
    fp=TR/f'{ds}__{seed}__{subset}__{name}.joblib'
    if fp.exists(): z=joblib.load(fp)
    else:
     clf=maker(seed); t=time.perf_counter(); clf.fit(X,y); train=time.perf_counter()-t; p=clf.predict(d['X_test'][:,ix]); score=clf.predict_proba(d['X_test'][:,ix])[:,1] if hasattr(clf,'predict_proba') else None; z={'dataset':ds,'seed':seed,'subset':subset,'classifier':name,'selected':len(ix),'train_s':train,**metrics(d['y_test'],p,score)}; atomic_joblib(z,fp)
    rows.append(z); print(ds,seed,subset,name,'F1',round(z['f1'],4),'MCC',round(z['mcc'],4))
   del X,y; gc.collect()
 del d; gc.collect(); ram('transfer released '+ds)
T=pd.DataFrame(rows); atomic_csv(T,DIR['results']/'classifier_transfer.csv')
display(T.groupby(['dataset','subset','classifier'])[['accuracy','recall','f1','specificity','mcc','pr_auc']].agg(['mean','std']).round(4))


In [ ]:
# CELL 7/9 — Drift robustness: tau sensitivity and false re-optimization
rows=[]
for ds in DATASETS:
 d=load_ds(ds)
 mask=joblib.load(FS/f'{ds}__TW2O__seed11.joblib')['mask'].astype(bool); Xref=d['X_val'][:,mask]; X=d['X_test'][:,mask].copy(); mu=Xref.mean(0); sd=Xref.std(0)+1e-6; rng=np.random.default_rng(2026)
 scenarios={'stationary':X.copy(),'benign_shift':X*1.03+rng.normal(0,.02,X.shape),'controlled_drift':X.copy()}; k=max(1,int(.35*X.shape[1])); cols=rng.choice(X.shape[1],k,False); scenarios['controlled_drift'][len(X)//2:,cols]+=.8
 for sc,Z in scenarios.items():
  for tau in CFG['tau_grid']:
   trig=[]; consec=0; scores=[]
   for st in range(0,len(Z)-CFG['drift_window']+1,CFG['drift_window']):
    score=float(np.mean(np.abs((Z[st:st+CFG['drift_window']].mean(0)-mu)/sd))); scores.append(score); consec=consec+1 if score>tau else 0
    if consec>=CFG['drift_consecutive']:trig.append(st+CFG['drift_window']);consec=0
   true=len(Z)//2 if sc=='controlled_drift' else None; false=sum(1 for q in trig if true is None or q<true); delay=(min([q for q in trig if q>=true],default=np.nan)-true) if true else np.nan
   rows.append({'dataset':ds,'scenario':sc,'tau':tau,'windows':len(scores),'mean_score':np.mean(scores),'triggers':len(trig),'false_triggers':false,'delay_samples':delay})
 del d,Xref,X,scenarios; gc.collect()
D=pd.DataFrame(rows); atomic_csv(D,DIR['results']/'drift_sensitivity.csv'); display(D)


In [ ]:
# CELL 8/9 — Security/resource accounting (claims kept separate from IDS accuracy)
# This cell intentionally does NOT claim that chaos statistics prove cryptographic security.
# It records deterministic resource proxies and prepares an auditable table. A standard AEAD baseline
# (e.g., Ascon) should be benchmarked on the actual target implementation before making cipher-level claims.
rows=[]
for ds in DATASETS:
 d=load_ds(ds)
 for seed in CFG['seeds']:
  r=joblib.load(FS/f'{ds}__TW2O__seed{seed}.joblib'); k=r['selected']; full=len(d['features']); h=CFG['tiny_hidden']
  rows.append({'dataset':ds,'seed':seed,'full_features':full,'selected_features':k,'feature_reduction':1-k/full,'full_dense_parameters':full*h+2*h+1,'selected_dense_parameters':k*h+2*h+1,'parameter_reduction':1-(k*h+2*h+1)/(full*h+2*h+1),'measured_test_inference_ms_per_sample':r['test']['inference_ms_per_sample']})
 del d; gc.collect()
Q=pd.DataFrame(rows); atomic_csv(Q,DIR['results']/'tinyml_resource_accounting.csv'); display(Q.groupby('dataset').agg(['mean','std']).round(4))
print('NOTE: Docker/host measurements are proxy trends, not physical MCU energy measurements. Do not label them as MCU power consumption.')


In [ ]:
# CELL 9/9 — Statistical analysis + paper-ready workbook + reproducibility bundle
from scipy.stats import wilcoxon
R=pd.read_csv(DIR['results']/'feature_selection_all_runs.csv'); out=[]
# Paired TW2O vs each stochastic baseline over matched seeds, metric by metric.
for ds in R.dataset.unique():
 for base in ['GA','PSO','DE','GWO','BWOA']:
  for metric in ['f1','mcc','recall','specificity','selected','selector_s']:
   a=R[(R.dataset==ds)&(R.method=='TW2O')].sort_values('seed')[metric].to_numpy(); b=R[(R.dataset==ds)&(R.method==base)].sort_values('seed')[metric].to_numpy()
   if len(a)==len(b) and len(a)>=5:
    try: stat,p=wilcoxon(a,b,zero_method='zsplit')
    except: stat,p=np.nan,np.nan
    out.append({'dataset':ds,'baseline':base,'metric':metric,'wilcoxon_stat':stat,'p_value':p,'tw2o_mean':a.mean(),'baseline_mean':b.mean()})
ST=pd.DataFrame(out); atomic_csv(ST,DIR['results']/'paired_statistics.csv')

xlsx=DIR['final']/'HEAL_R2_clean_results.xlsx'
with pd.ExcelWriter(xlsx,engine='openpyxl') as w:
 R.to_excel(w,'FS_all_runs',index=False)
 pd.read_csv(DIR['results']/'classifier_transfer.csv').to_excel(w,'Classifier_transfer',index=False)
 pd.read_csv(DIR['results']/'drift_sensitivity.csv').to_excel(w,'Drift',index=False)
 pd.read_csv(DIR['results']/'tinyml_resource_accounting.csv').to_excel(w,'Resources',index=False)
 pd.read_csv(DIR['results']/'pre_experiment_feature_audit.csv').to_excel(w,'Leakage_audit',index=False)
 ST.to_excel(w,'Statistics',index=False)

bundle=DIR['final']/'HEAL_R2_reproducibility_bundle.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
 z.write(RUN/'config.json',arcname='config.json')
 for f in DIR['results'].glob('*.csv'): z.write(f,arcname='results/'+f.name)
print('Workbook:',xlsx); print('Bundle:',bundle); print('[DONE] Use measured outputs only; do not preserve old headline numbers if the new experiments differ.')
